
# AISC Steel Specifications - Implementation Notes

Python implementations of select AISC specification tables and
demonstrations of the `civilpy.structural.steel` module.

All design provisions reference:

- AISC. *Steel Construction Manual*, 16th Ed.
- AISC 360. *Specification for Structural Steel Buildings*.

Refer to the current AISC specifications for authoritative design provisions.


Originally based on the 14th edition, subject to updates.

The AISC Steel manual is broken up into four general sections.  The first
outlines common steel shapes, this can be pretty directly translated to
python using civilpy.

In [1]:
from civilpy.structural.steel import W, steel_tables

w12x58 = W("W12X58")
print(w12x58.area)

In [14]:
# Common design task - find the lightest W shape that meets the criteria S_x > 700
lightest_section = steel_tables[(steel_tables['Type'] == 'W') & (steel_tables['Sx'] >= 700)].sort_values('W').iloc[0]

In [16]:
lightest_section['AISC_Manual_Label'], lightest_section['Sx']

The second section outlines general design principles utilized in steel  
construction for various element types and things like welds, connections,  
etc.  

The third section is made up of specifications. This includes the three  
primary specifications that make up this section. "Specification for  
Structural Steel Buildings" (16.1), "Specification for Structural Joints  
Using High-Strength Bolts" (16.2), and "Code of Standard Practice for Steel  
Buildings and Bridges" (16.3).  

The fourth section is made up of Appendixes and miscellaneous design tools.

In [19]:
# See above, might go through and provide further examples here later.

In [ ]:
# AISC Table D3.1 - Shear Lag Factors for Connections to Tension Members
# Reference: AISC 360-22 Chapter D
#
# For cases where load is transmitted through some but not all cross-sectional elements
# (Case 2), the shear lag factor is computed as:
#   U = 1 - x_bar / L
# where:
#   x_bar = distance from shear plane to centroid of the connected element (in.)
#   L     = length of connection in the direction of load (in.)
#
# For specific member/connection configurations, use the tabulated U values below.

AISC_D3_1_SHEAR_LAG_FACTORS = {
    # Case 1: Load transmitted directly to each cross-sectional element
    # (all elements connected by fasteners or welds)
    "case_1": {
        "description": "All elements connected; load distributed to each element",
        "U": 1.0,
        "applies_to": "W, M, S, HP shapes; structural tees; angles — when all elements connected",
    },
    # Case 2: Load not through all elements — use U = 1 - x_bar/L (computed per connection)
    "case_2": {
        "description": "Some but not all cross-sectional elements connected",
        "U": "1 - x_bar / L",  # Formula; compute per connection
        "applies_to": "All tension members except plates and HSS",
        "min_U_override": (
            "For W, M, S, HP or tees: U >= A_connected / A_gross (gross area of connected element / total)"
        ),
    },
    # Case 3: Plates — longitudinal welds only
    "case_3_lw_ge_2w": {"description": "Plate with longitudinal welds, l >= 2w", "U": 1.00},
    "case_3_lw_ge_1_5w": {"description": "Plate with longitudinal welds, 2w > l >= 1.5w", "U": 0.87},
    "case_3_lw_ge_1_0w": {"description": "Plate with longitudinal welds, 1.5w > l >= w", "U": 0.75},
    # Case 4: W-shapes, flanges connected with >=3 fasteners per line, bf >= 2/3 d
    "case_4_bf_ge_2_3d": {"description": "W-shape flange-connected, bf >= 2/3 d, >=3 fasteners/line", "U": 0.90},
    # Case 5: W-shapes, flanges connected with >=3 fasteners per line, bf < 2/3 d
    "case_5_bf_lt_2_3d": {"description": "W-shape flange-connected, bf < 2/3 d, >=3 fasteners/line", "U": 0.85},
    # Case 6: W-shapes, web connected only, >=4 fasteners per line
    "case_6_web_4plus": {"description": "W-shape web-only connection, >=4 fasteners/line", "U": 0.70},
    # Case 7: Single/double angles
    "case_7_4plus_fasteners": {"description": "Single/double angle, >=4 fasteners/line", "U": 0.80},
    "case_7_23_fasteners": {"description": "Single/double angle, 2-3 fasteners/line", "U": 0.60},
    # Case 8: HSS — round, l >= 1.3D
    "case_8_round_l_ge_1_3D": {"description": "Round HSS with single concentric gusset, l >= 1.3D", "U": 1.00},
    "case_8_round_l_ge_D": {"description": "Round HSS with single concentric gusset, D <= l < 1.3D", "U": 0.90},
    # Case 8: HSS — rectangular, connected on all 4 sides or 3 sides
    "case_8_rect_all_walls": {"description": "Rectangular HSS connected through all 4 walls", "U": 1.00},
    "case_8_rect_3_walls_long": {"description": "Rectangular HSS connected through 3 walls, l >= 1.3H", "U": 0.90},
    "case_8_rect_3_walls_short": {"description": "Rectangular HSS connected through 3 walls, l < 1.3H", "U": 0.85},
}


def get_shear_lag_factor(case_key, x_bar=None, L=None):
    """Return the shear lag factor U for a given AISC Table D3.1 case.

    For Case 2, supply x_bar and L to compute U = 1 - x_bar/L.

    Args:
        case_key (str): Key from AISC_D3_1_SHEAR_LAG_FACTORS.
        x_bar (float, optional): Centroid distance for Case 2.
        L (float, optional): Connection length for Case 2.

    Returns:
        float: Shear lag factor U.
    """
    entry = AISC_D3_1_SHEAR_LAG_FACTORS[case_key]
    U = entry["U"]
    if U == "1 - x_bar / L":
        if x_bar is None or L is None:
            raise ValueError("Case 2 requires x_bar and L to compute U = 1 - x_bar/L")
        return 1.0 - x_bar / L
    return U


In [ ]:
# AISC Table E7.1 - Effective Width Imperfection Adjustment Factor c1
# Reference: AISC 360-22 Section E7
# Used in Equations E7-2 and E7-3 for slender element members
#
#   b_e = b * (1 - c1*sqrt(F_el/F_n)) * sqrt(F_el/F_n)  [Eq E7-3]
#   c2  = (1 - sqrt(1 - 4*c1)) / (2*c1)                 [Eq E7-4]

AISC_E7_1_IMPERFECTION_FACTORS = {
    # element_type: (c1, c2_formula)
    # c2 is derived from c1 via Eq. E7-4; values below match Table E7.1 directly
    "uniform_compression_flanges_webs": {
        "description": "Uniformly compressed flanges and webs (outstanding legs not included)",
        "c1": 0.18,
        "c2": 1.31,  # = (1 - sqrt(1 - 4*0.18)) / (2*0.18)
    },
    "webs_in_flexure": {
        "description": "Webs in flexural compression",
        "c1": 0.34,
        "c2": 1.86,  # = (1 - sqrt(1 - 4*0.34)) / (2*0.34)
    },
}


def get_effective_width(b, c1, F_el, F_n, lambda_r, lambda_val):
    """Compute effective width per AISC 360-22 Equations E7-2 and E7-3.

    Args:
        b (float): Width of the slender element (in.).
        c1 (float): Imperfection factor from Table E7.1.
        F_el (float): Elastic local buckling stress (ksi) from Eq. E7-5.
        F_n (float): Flexural buckling stress (ksi) from the applicable section.
        lambda_r (float): Limiting width-to-thickness ratio from Table B4.1a.
        lambda_val (float): Actual width-to-thickness ratio b/t.

    Returns:
        float: Effective width b_e (in.).
    """
    import math
    threshold = lambda_r * math.sqrt(F_y / F_n)  # noqa: F821 — F_y from outer scope
    if lambda_val <= threshold:
        return b  # Eq. E7-2
    else:
        ratio = math.sqrt(F_el / F_n)
        return b * (1 - c1 * ratio) * ratio  # Eq. E7-3
